# 🔢 LAB 1: Rakamları Okuyan Yapay Zekâ

> **Görev:** Bilgisayara el yazısı rakamları tanımayı öğreteceğiz. En sonunda **sen kendi rakamını çizeceksin**, yapay zekâ tahmin edecek! ✍️🤖

Postanelerdeki makineler zarflardaki posta kodlarını, bankalar çeklerdeki yazıları, telefonun kamerası da fotoğraftaki yazıları bu yöntemle okur.
Buna **görüntü sınıflandırma** (*image classification* — bir resmin hangi gruba ait olduğunu bulmak) denir.

### 🗺️ Laboratuvar Haritası
1. 🔍 Veriyi tanı: Bilgisayar bir resmi nasıl "görür"?
2. ✂️ Veriyi eğitim ve test olarak ayır
3. 🏋️ İlk modelimizi eğit (K-En Yakın Komşu)
4. 📝 Modele sınav yap ve hatalarını incele
5. 🏁 Model yarışması: Hangi algoritma daha iyi?
6. ✍️ **Kendi rakamını çiz, yapay zekâ tahmin etsin!**

> ⏱️ Süre: yaklaşık 2 ders saati &nbsp;|&nbsp; 🎮 Hücreleri sırayla **Shift + Enter** ile çalıştır.

In [ ]:
# Gerekli kütüphaneleri çağıralım
import numpy as np                       # sayılarla hesap
import pandas as pd                      # tablolar
import matplotlib.pyplot as plt          # grafikler
import seaborn as sns                    # güzel grafikler
from sklearn.datasets import load_digits # scikit-learn içindeki hazır rakam veri seti (dataset)

sns.set_theme(style="white")             # grafik teması
np.random.seed(0)                        # seed (tohum): rastgele işlemler her seferinde aynı sonucu versin
print("Hazırız! 🚀")

---
## 1️⃣ Veriyi Tanıyalım 🔍

Kullanacağımız veri setinde, gerçek insanların kâğıda yazdığı **1797 tane rakam resmi** var.
Her resim **8 piksel × 8 piksel** boyutunda küçücük siyah-beyaz bir resim.

In [ ]:
rakamlar = load_digits()     # veri setini yükle

print("Resim sayısı:", len(rakamlar.images))
print("Bir resmin boyutu:", rakamlar.images[0].shape, "→ 8 satır, 8 sütun piksel")
print("Etiketler (cevaplar):", rakamlar.target[:20], "...")     # target (hedef): her resmin hangi rakam olduğu

### 🤖 Bilgisayar bir resmi nasıl görür?
Biz bir resme bakınca şekil görürüz. Bilgisayar ise sadece **sayılar** görür!
Her piksel **0 ile 16** arasında bir sayıdır: **0 = boş (siyah)**, **16 = tam dolu (beyaz)**.

In [ ]:
ornek = rakamlar.images[0]          # ilk resmi alalım

fig, eksen = plt.subplots(1, 2, figsize=(10, 4.5))

# Sol taraf: Bilgisayarın gördüğü → SAYILAR
eksen[0].imshow(np.zeros((8, 8)), cmap="gray_r", vmin=0, vmax=1)   # boş beyaz zemin
for satir in range(8):
    for sutun in range(8):
        deger = int(ornek[satir, sutun])
        eksen[0].text(sutun, satir, deger, ha="center", va="center",   # ha/va: yatay/dikey hizalama (horizontal/vertical alignment)
                      fontsize=12, color="red" if deger > 8 else "gray")
eksen[0].set_title("Bilgisayarın gördüğü: 64 sayı")
eksen[0].set_xticks(range(8)); eksen[0].set_yticks(range(8))         # xticks/yticks: eksen üzerindeki işaretler

# Sağ taraf: Bizim gördüğümüz → RESİM
eksen[1].imshow(ornek, cmap="gray")       # imshow (image show): sayıları renk olarak göster
eksen[1].set_title(f"Bizim gördüğümüz: {rakamlar.target[0]} rakamı")
eksen[1].axis("off")
plt.show()

In [ ]:
# Veri setinden rastgele 30 rakamı görelim
fig, eksenler = plt.subplots(3, 10, figsize=(14, 4.5))
secilenler = np.random.choice(len(rakamlar.images), 30, replace=False)   # random.choice: rastgele 30 tane seç (replace=False → aynısını iki kez seçme)

for eksen, i in zip(eksenler.ravel(), secilenler):     # ravel: 3x10'luk eksen tablosunu tek sıraya diz
    eksen.imshow(rakamlar.images[i], cmap="gray")
    eksen.set_title(str(rakamlar.target[i]), fontsize=14, color="tab:blue")
    eksen.axis("off")
plt.suptitle("Veri setinden örnekler (üstteki mavi sayı = doğru cevap)", fontsize=14)   # suptitle (super title): genel başlık
plt.tight_layout()
plt.show()

> 🤔 **Gözlemle:** Bazı rakamları sen bile zor tanıyorsun, değil mi? Herkesin el yazısı farklı! Yapay zekâ bu çeşitliliğe rağmen öğrenmek zorunda.

In [ ]:
# Her rakamdan kaç tane var? Dengeli bir veri seti mi?
plt.figure(figsize=(8, 3.5))
sns.countplot(x=rakamlar.target, color="tab:blue")      # countplot (sayım grafiği)
plt.title("Her rakamdan kaç örnek var?")
plt.xlabel("Rakam")
plt.ylabel("Örnek sayısı")
plt.show()

> ✅ Her rakamdan yaklaşık 180 tane var. Buna **dengeli veri seti** (*balanced dataset*) denir. Eğer 1000 tane "1" ve sadece 10 tane "7" olsaydı, model "7"yi iyi öğrenemezdi!

### 🧙 Sihirli Deney: "Ortalama Rakam"
Tüm "0"ları üst üste koyup ortalamasını alırsak ne olur? NumPy ile tek satırda yapabiliriz!

In [ ]:
fig, eksenler = plt.subplots(1, 10, figsize=(15, 2.2))
for rakam in range(10):
    bu_rakamin_resimleri = rakamlar.images[rakamlar.target == rakam]   # sadece bu rakama ait resimleri filtrele
    ortalama_resim = bu_rakamin_resimleri.mean(axis=0)                # axis=0 → resimlerin üzerinden ortalama al (piksel piksel)
    eksenler[rakam].imshow(ortalama_resim, cmap="magma")              # magma: ateş gibi bir renk haritası (colormap)
    eksenler[rakam].set_title(f"Ortalama {rakam}")
    eksenler[rakam].axis("off")
plt.show()

> 🔥 Parlak bölgeler, o rakamı yazan **çoğu kişinin** kalemini geçirdiği yerler. Her rakamın kendine özgü bir "parmak izi" var! Modelimiz de tam olarak bu farklılıkları öğrenecek.

### 📏 Resmi bir sıraya dizmek
Makine öğrenmesi modelleri tablo şeklinde veri sever: **her satır bir örnek, her sütun bir özellik.**
Bu yüzden 8×8'lik resmi, satırları uç uca ekleyerek **64 sayılık uzun bir sıraya** çeviririz. Buna **düzleştirme** (*flatten*) denir.

```
 8x8 resim           →        64 sayılık tek satır
 [ satır 1 ]
 [ satır 2 ]         →   [ satır1 | satır2 | satır3 | ... | satır8 ]
 [  ...    ]
```
Veri setinde bu iş bizim için zaten yapılmış: `rakamlar.data`

In [ ]:
X = rakamlar.data       # özellikler (features): her resim 64 sayı
y = rakamlar.target     # etiketler (labels): her resmin doğru cevabı

print("X'in şekli:", X.shape, "→ 1797 resim, her biri 64 özellik (piksel)")
print("y'nin şekli:", y.shape)
print("\nİlk resmin 64 sayısı:\n", X[0].astype(int))

---
## 2️⃣ Eğitim ve Test Verisi ✂️

Hatırla: Modeli **hiç görmediği** resimlerle sınamazsak, ezberleyip ezberlemediğini anlayamayız!
- 📖 **Eğitim verisi** (%75): Model bu resimlerle öğrenecek.
- 📝 **Test verisi** (%25): Kilitli bir kasada saklayacağız, en sonda sınav için açacağız.

In [ ]:
from sklearn.model_selection import train_test_split

X_egitim, X_test, y_egitim, y_test = train_test_split(
    X, y,
    test_size=0.25,      # test_size (test boyutu): verinin %25'i test için
    random_state=42,     # random_state (rastgelelik durumu): her çalıştırmada aynı şekilde bölünsün
    stratify=y           # stratify (katmanlara ayır): her rakamdan eğitimde de testte de eşit oranda olsun
)

print(f"📖 Eğitim: {len(X_egitim)} resim")
print(f"📝 Test  : {len(X_test)} resim")

---
## 3️⃣ İlk Modelimiz: K-En Yakın Komşu (KNN) 🏘️

**KNN** (*K-Nearest Neighbors* — K tane en yakın komşu) en sezgisel makine öğrenmesi algoritmasıdır:

1. Yeni bir resim gelir. ❓
2. Model, eğitimdeki **tüm** resimlerle tek tek karşılaştırır: "Hangisine en çok benziyor?"
3. En çok benzeyen **K tane** resmi seçer (örneğin K=3).
4. Bu komşuların **çoğunluğu** hangi rakamsa, cevap odur! 🗳️

> 🧮 **İki resim ne kadar benzer?** 64 pikselin her birindeki farkı hesaplayıp toplarız. Fark ne kadar küçükse resimler o kadar benzer. Buna **mesafe** (*distance*) diyoruz.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier   # Classifier (sınıflandırıcı): örnekleri gruplara ayıran model
from sklearn.metrics import accuracy_score           # accuracy (doğruluk) hesaplamak için

knn = KNeighborsClassifier(n_neighbors=3)   # 1️⃣ Model oluştur: 3 komşuya bak
knn.fit(X_egitim, y_egitim)                 # 2️⃣ Eğit (fit): KNN aslında eğitim resimlerini hafızasına alır
tahminler = knn.predict(X_test)             # 3️⃣ Tahmin et (predict): test resimleri için cevap üret

dogruluk = accuracy_score(y_test, tahminler)
print(f"🎯 Test doğruluğu: %{dogruluk * 100:.2f}")
print(f"   {len(y_test)} resimden {int((tahminler == y_test).sum())} tanesini doğru bildi!")

> 🤯 Model hiç görmediği resimlerin **%98'inden fazlasını** doğru tahmin etti! Ve biz ona "1 dik bir çizgidir", "0 yuvarlaktır" gibi **tek bir kural bile söylemedik.** Sadece örnekler verdik.

### 👀 Modelin aklından geçenleri görelim
Bir test resmi seçip, modelin onu **hangi komşulara benzettiğine** bakalım:

In [ ]:
test_no = 7    # 🎯 Bu sayıyı değiştirip farklı resimleri dene! (0 ile 449 arası)

mesafeler, komsu_indeksleri = knn.kneighbors([X_test[test_no]])   # kneighbors: en yakın komşuları ve mesafelerini bul

fig, eksenler = plt.subplots(1, 4, figsize=(11, 3))
eksenler[0].imshow(X_test[test_no].reshape(8, 8), cmap="gray")    # reshape: 64 sayıyı tekrar 8x8 resme çevir
eksenler[0].set_title(f"Soru resmi\nTahmin: {tahminler[test_no]}", color="tab:blue", fontsize=13)

for i, (komsu, mesafe) in enumerate(zip(komsu_indeksleri[0], mesafeler[0])):
    eksenler[i + 1].imshow(X_egitim[komsu].reshape(8, 8), cmap="gray")
    eksenler[i + 1].set_title(f"{i + 1}. komşu: {y_egitim[komsu]}\nmesafe: {mesafe:.1f}", fontsize=11)
for e in eksenler:
    e.axis("off")
plt.suptitle("Model, soru resmine en çok benzeyen 3 eğitim resmine bakıp oylama yaptı", y=1.08)
plt.show()

---
## 4️⃣ Sınav Kâğıdını İnceleyelim 📝

Test resimlerinden bazılarını, modelin tahminleriyle birlikte görelim. **Yeşil** = doğru ✅, **Kırmızı** = yanlış ❌

In [ ]:
fig, eksenler = plt.subplots(3, 10, figsize=(14, 5))
for eksen, i in zip(eksenler.ravel(), range(30)):
    dogru_mu = tahminler[i] == y_test[i]
    eksen.imshow(X_test[i].reshape(8, 8), cmap="gray")
    eksen.set_title(f"Tahmin: {tahminler[i]}", color="green" if dogru_mu else "red", fontsize=11)
    eksen.axis("off")
plt.suptitle("Modelin sınav kâğıdı (ilk 30 soru)", fontsize=14)
plt.tight_layout()
plt.show()

### 🧩 Karışıklık Matrisi (*Confusion Matrix*)
Model hangi rakamları **birbirine karıştırıyor?** Bunu görmek için harika bir araç var: **karışıklık matrisi**.
- **Satırlar:** Gerçek rakam
- **Sütunlar:** Modelin tahmini
- **Köşegen** (sol üstten sağ alta çapraz çizgi): Doğru tahminler ✅
- **Köşegen dışı:** Hatalar ❌ (Örneğin satır 8, sütun 1'deki sayı = "aslında 8 olup modelin 1 sandığı resim sayısı")

In [ ]:
from sklearn.metrics import confusion_matrix

karisiklik = confusion_matrix(y_test, tahminler)    # gerçek değerler ile tahminleri karşılaştır

plt.figure(figsize=(8, 6.5))
sns.heatmap(karisiklik, annot=True, fmt="d", cmap="Blues", cbar=False)   # annot: kutulara sayıyı yaz, fmt="d": tam sayı formatı
plt.xlabel("Modelin TAHMİNİ", fontsize=12)
plt.ylabel("GERÇEK rakam", fontsize=12)
plt.title("Karışıklık Matrisi", fontsize=14)
plt.show()

In [ ]:
# Modelin yaptığı TÜM hataları görelim
hatalar = np.where(tahminler != y_test)[0]      # where (nerede): tahminin yanlış olduğu yerlerin sıra numaraları
print(f"Toplam hata sayısı: {len(hatalar)}")

fig, eksenler = plt.subplots(1, len(hatalar), figsize=(2.2 * len(hatalar), 2.8))
eksenler = np.atleast_1d(eksenler)              # tek hata olsa bile liste gibi davranabilsin
for eksen, i in zip(eksenler, hatalar):
    eksen.imshow(X_test[i].reshape(8, 8), cmap="gray")
    eksen.set_title(f"Gerçek: {y_test[i]}\nTahmin: {tahminler[i]}", color="red")
    eksen.axis("off")
plt.show()

> 🤔 **Tartışalım:** Bu hataların bazılarını sen de yapar mıydın? Hata yapılan resimler genelde **gerçekten kötü yazılmış** rakamlardır. Yapay zekâ da biz insanlar gibi kötü el yazısında zorlanıyor! 😅

### 🎛️ K kaç olmalı?
K=3 seçtik ama bu en iyisi mi? Hadi 1'den 15'e kadar tüm K değerlerini deneyelim!

In [ ]:
k_degerleri = range(1, 16)
dogruluklar = []

for k in k_degerleri:
    model = KNeighborsClassifier(n_neighbors=k).fit(X_egitim, y_egitim)   # fit(...) modeli geri döndürür, o yüzden zincirleyebiliriz
    dogruluklar.append(accuracy_score(y_test, model.predict(X_test)))

en_iyi_k = k_degerleri[int(np.argmax(dogruluklar))]    # argmax: en büyük değerin sıra numarası

plt.figure(figsize=(9, 3.8))
plt.plot(k_degerleri, np.array(dogruluklar) * 100, marker="o", linewidth=2)
plt.axvline(en_iyi_k, color="green", linestyle="--", label=f"En iyi K = {en_iyi_k}")
plt.xlabel("K (bakılan komşu sayısı)")
plt.ylabel("Test doğruluğu (%)")
plt.title("Komşu sayısı başarıyı nasıl etkiliyor?")
plt.xticks(k_degerleri)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

> 💡 K değeri gibi, **modeli kurmadan önce bizim seçtiğimiz** ayarlara **hiperparametre** (*hyperparameter* — modelin öğrenmediği, bizim belirlediğimiz ayar) denir. En iyi ayarı bulmak, makine öğrenmesi mühendislerinin en önemli işlerinden biridir!

---
## 5️⃣ Büyük Model Yarışması 🏁

KNN tek algoritma değil! Scikit-learn'deki diğer güçlü algoritmaları da yarıştıralım:

| Yarışmacı | Nasıl düşünür? |
|-----------|----------------|
| 🏘️ **KNN** (*K-Nearest Neighbors*) | "Sana en çok benzeyen komşularına bakarım." |
| 🌳 **Karar Ağacı** (*Decision Tree*) | "Evet/hayır soruları sorarak cevaba ulaşırım: 'Ortadaki piksel dolu mu?'" |
| 🌲🌲 **Rastgele Orman** (*Random Forest*) | "Yüzlerce farklı karar ağacı kurar, hepsine oy verdiririm." |
| ✂️ **Destek Vektör Makinesi** (*SVM — Support Vector Machine*) | "Rakam grupları arasına mümkün olan en geniş sınırı çizerim." |
| 📈 **Lojistik Regresyon** (*Logistic Regression*) | "Her piksele bir önem puanı veririm, puanları toplarım." |

Hepsi **aynı 3 adımı** kullanıyor: oluştur → `fit` → `predict`. Scikit-learn'ün güzelliği bu!

In [ ]:
import time
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC                          # SVC: Support Vector Classifier (Destek Vektör Sınıflandırıcısı)
from sklearn.linear_model import LogisticRegression

yarismacilar = {
    "KNN": KNeighborsClassifier(n_neighbors=en_iyi_k),
    "Karar Ağacı": DecisionTreeClassifier(random_state=0),
    "Rastgele Orman": RandomForestClassifier(n_estimators=300, random_state=0),   # n_estimators: ormandaki ağaç sayısı
    "SVM": SVC(gamma=0.001, C=10),                   # gamma ve C: SVM'in hiperparametreleri
    "Lojistik Regresyon": LogisticRegression(max_iter=5000),   # max_iter (maksimum tekrar): öğrenmek için en fazla kaç adım atsın
}

sonuclar = []
for isim, model in yarismacilar.items():
    baslangic = time.time()
    model.fit(X_egitim, y_egitim)
    sure = time.time() - baslangic
    dogruluk = accuracy_score(y_test, model.predict(X_test))
    sonuclar.append({"Model": isim, "Doğruluk (%)": round(dogruluk * 100, 2), "Eğitim süresi (sn)": round(sure, 3)})
    print(f"✅ {isim:<20} → %{dogruluk * 100:.2f}")

sonuc_tablosu = pd.DataFrame(sonuclar).sort_values("Doğruluk (%)", ascending=False).reset_index(drop=True)
sonuc_tablosu

In [ ]:
plt.figure(figsize=(9, 4))
renkler = ["gold", "silver", "#cd7f32", "lightsteelblue", "lightsteelblue"]   # altın, gümüş, bronz 🥇🥈🥉
cubuklar = plt.barh(sonuc_tablosu["Model"][::-1], sonuc_tablosu["Doğruluk (%)"][::-1], color=renkler[::-1])  # [::-1] → listeyi ters çevir
plt.bar_label(cubuklar, fmt="%.1f%%", padding=4)     # bar_label: çubukların ucuna değeri yaz
plt.xlim(80, 101)
plt.xlabel("Test doğruluğu (%)")
plt.title("Model Yarışması Sonuçları")
plt.show()

> 🌳 **Karar ağacı neden geride kaldı?** Tek bir ağaç, eğitim verisini **ezberlemeye** (*overfitting*) çok yatkındır. Rastgele Orman ise yüzlerce ağacın ortak kararını aldığı için çok daha sağlamdır. "Bir elin nesi var, iki elin sesi var!" 🌲🌲🌲

---
## 6️⃣ ✍️ Şimdi Sıra Sende: Kendi Rakamını Çiz!

Artık yapay zekâmız hazır. Hadi onu gerçek bir testten geçirelim: **senin el yazınla!**

Bunun için önce iki yardımcı **fonksiyon** (*function*) yazalım:
1. `resmi_hazirla`: Senin çizdiğin büyük resmi, veri setindeki gibi **8×8'lik** küçük resme çevirir.
2. `tahmin_goster`: Resmi modele verir ve sonucu güzel bir grafikle gösterir.

Tahmin için **Rastgele Orman** modelini kullanacağız. Çünkü bize sadece cevabı değil, **ağaçların oylarını** da gösterebiliyor. Böylece modelin ne kadar **emin** olduğunu görebiliriz!

In [ ]:
from PIL import Image        # PIL (Python Imaging Library): resim işleme kütüphanesi

# Son modelimizi TÜM veriyle eğitelim (ne kadar çok örnek, o kadar iyi öğrenme!)
son_model = RandomForestClassifier(n_estimators=500, random_state=0)
son_model.fit(X, y)


def resmi_hazirla(resim):
    """Herhangi bir rakam resmini, veri setindeki 8x8 formatına çevirir."""
    gri = np.array(resim.convert("L"), dtype=float)     # convert("L"): resmi gri tonlamaya (siyah-beyaz) çevir. 0=siyah, 255=beyaz
    if gri.mean() > 127:                                # Zemin açık renkliyse (beyaz kâğıda siyah kalem)...
        gri = 255 - gri                                 # ...renkleri ters çevir. Veri setinde rakamlar siyah zemin üzerine beyaz!

    # 1) Rakamın etrafındaki boşlukları kes (kırp)
    satirlar, sutunlar = np.where(gri > 50)
    if len(satirlar) == 0:
        return np.zeros(64)                             # boş resim → hepsi sıfır
    gri = gri[satirlar.min():satirlar.max() + 1, sutunlar.min():sutunlar.max() + 1]

    # 2) Rakamı 32x32'lik bir kareye sığdır (en-boy oranını bozmadan, ortala)
    yukseklik, genislik = gri.shape
    olcek = 32 / max(yukseklik, genislik)
    yeni_boyut = (max(1, round(genislik * olcek)), max(1, round(yukseklik * olcek)))
    kucuk = np.array(Image.fromarray(gri.astype(np.uint8)).resize(yeni_boyut, Image.LANCZOS), dtype=float)  # LANCZOS: kaliteli küçültme yöntemi
    kare = np.zeros((32, 32))
    ust, sol = (32 - kucuk.shape[0]) // 2, (32 - kucuk.shape[1]) // 2
    kare[ust:ust + kucuk.shape[0], sol:sol + kucuk.shape[1]] = kucuk

    # 3) Kalemin kalınlığını veri setine benzet: en parlak ~300 pikseli "mürekkep" say
    esik = np.sort(kare.ravel())[::-1][min(300, int((kare > 20).sum())) - 1]
    murekkep = kare >= max(esik, 20)

    # 4) 32x32'yi 4x4'lük bloklara böl, her bloktaki mürekkepli pikselleri say → 8x8, değerler 0-16
    #    (Veri setinin yaratıcıları da tam olarak böyle yapmış!)
    return murekkep.reshape(8, 4, 8, 4).sum(axis=(1, 3)).ravel().astype(float)


def tahmin_goster(ozellikler, baslik="Senin rakamın"):
    """8x8'lik resmi modele verir ve ağaçların oylarını grafikle gösterir."""
    oylar = son_model.predict_proba([ozellikler])[0]   # predict_proba (olasılık tahmini): her rakam için oy oranı
    tahmin = int(np.argmax(oylar))

    fig, eksen = plt.subplots(1, 2, figsize=(10, 3.5), gridspec_kw={"width_ratios": [1, 2]})
    eksen[0].imshow(ozellikler.reshape(8, 8), cmap="gray")
    eksen[0].set_title(baslik)
    eksen[0].axis("off")

    renkler = ["tab:green" if r == tahmin else "lightgray" for r in range(10)]
    eksen[1].bar(range(10), oylar * 100, color=renkler)
    eksen[1].set_xticks(range(10))
    eksen[1].set_ylim(0, 100)
    eksen[1].set_xlabel("Rakam")
    eksen[1].set_ylabel("Ağaçların oyu (%)")
    eksen[1].set_title(f"Tahmin: {tahmin}  (ağaçların %{oylar[tahmin] * 100:.0f}'i bu fikirde)")
    plt.tight_layout()
    plt.show()
    return tahmin

print("Fonksiyonlar hazır! ✅")

### 🅰️ Yöntem 1: Klavyeyle çiz (her yerde çalışır)
Aşağıdaki ızgarada `#` işareti **mürekkep**, `.` işareti **boş kâğıt** demek.
`#`'leri değiştirerek kendi rakamını çiz ve hücreyi çalıştır! Örnekte bir **2** var.

> 💡 Rakamı **uzun boylu** çiz (veri setindeki rakamlar genişliğinden daha uzun). Satır ve sütun sayısını istediğin gibi değiştirebilirsin, hepsi aynı uzunlukta olsun yeter.

In [ ]:
from PIL import ImageFilter    # ImageFilter: resimlere filtre (bulanıklaştırma vb.) uygulamak için

cizim = """
.###..
#...#.
....#.
....#.
...#..
..#...
.#....
#.....
#.....
#####.
"""

# 1) Metni sayı tablosuna çevir: '#' → 255 (beyaz mürekkep), '.' → 0 (siyah zemin)
satirlar = cizim.strip().split("\n")            # strip: baştaki/sondaki boşlukları at, split: satırlara böl
izgara = np.array([[255 if p == "#" else 0 for p in satir] for satir in satirlar], dtype=np.uint8)

# 2) Izgarayı büyüt ve biraz bulanıklaştır → kalemle çizilmiş gibi yumuşak bir resim olsun
buyuk = Image.fromarray(izgara).resize((izgara.shape[1] * 10, izgara.shape[0] * 10), Image.NEAREST)  # NEAREST: pikselleri kare kare büyüt
buyuk = buyuk.filter(ImageFilter.GaussianBlur(9))   # GaussianBlur: yumuşak bulanıklaştırma filtresi

# 3) Diğer yöntemlerle AYNI hazırlık fonksiyonundan geçir ve tahmin et
tahmin_goster(resmi_hazirla(buyuk), "Klavyeyle çizdiğin rakam");

### 🅱️ Yöntem 2: Fareyle (ya da parmağınla) çiz! 🖱️ *(Sadece Google Colab'da çalışır)*

Hücreyi çalıştır → Siyah kutunun içine **büyük ve kutuyu dolduracak şekilde** bir rakam çiz → **"Tahmin Et 🤖"** düğmesine bas.
Tekrar denemek için hücreyi yeniden çalıştır.

> 💡 **İpuçları:** Rakamı kutunun ortasına, büyük ve tek parça çiz. 7'nin ortasına çizgi, 1'in altına taban çizgisi koyma. Veri setindeki kişiler öyle yazmamış! 😉

In [ ]:
import base64, io                         # base64: resmi metin olarak taşımak için kodlama yöntemi; io: bellekte dosya gibi çalışma
from IPython.display import HTML, display

CIZIM_TUVALI = """
<div style="font-family:sans-serif">
  <canvas id="tuval" width="280" height="280"
          style="background:black; border-radius:12px; touch-action:none; cursor:crosshair"></canvas><br>
  <button id="temizle" style="font-size:16px; margin:6px">🧽 Temizle</button>
  <button id="bitti" style="font-size:16px; margin:6px; background:#2e7d32; color:white; border:none; padding:6px 14px; border-radius:6px">Tahmin Et 🤖</button>
</div>
<script>
  var tuval = document.getElementById('tuval');
  var ctx = tuval.getContext('2d');
  ctx.fillStyle = 'black'; ctx.fillRect(0, 0, 280, 280);
  ctx.strokeStyle = 'white'; ctx.lineWidth = 16; ctx.lineCap = 'round'; ctx.lineJoin = 'round';
  var ciziyor = false;
  function konum(e) { var r = tuval.getBoundingClientRect(); return [e.clientX - r.left, e.clientY - r.top]; }
  tuval.addEventListener('pointerdown', function(e) { ciziyor = true; var p = konum(e); ctx.beginPath(); ctx.moveTo(p[0], p[1]); });
  tuval.addEventListener('pointermove', function(e) { if (!ciziyor) return; var p = konum(e); ctx.lineTo(p[0], p[1]); ctx.stroke(); });
  window.addEventListener('pointerup', function() { ciziyor = false; });
  document.getElementById('temizle').onclick = function() { ctx.fillRect(0, 0, 280, 280); };
  var cizimSonucu = new Promise(function(resolve) {
    document.getElementById('bitti').onclick = function() { resolve(tuval.toDataURL('image/png')); };
  });
</script>
"""

def ciz_ve_tahmin_et():
    try:
        from google.colab.output import eval_js   # eval_js: Colab'da tarayıcıdaki JavaScript kodundan sonuç almak için
    except ImportError:
        print("⚠️ Çizim tuvali sadece Google Colab'da çalışır. Yöntem 1'i (klavyeyle çizim) kullanabilirsin.")
        return
    display(HTML(CIZIM_TUVALI))               # tuvali ekranda göster
    veri = eval_js("cizimSonucu")             # kullanıcı "Tahmin Et" düğmesine basana kadar bekle
    resim = Image.open(io.BytesIO(base64.b64decode(veri.split(",")[1])))   # gelen metni tekrar resme çevir
    tahmin_goster(resmi_hazirla(resim), "Senin çizimin (8x8 hali)")

ciz_ve_tahmin_et()

### 🅲 Yöntem 3: Kâğıda yaz, fotoğrafını yükle 📸 *(Sadece Google Colab'da çalışır)*
Beyaz bir kâğıda **kalın siyah bir kalemle** büyük bir rakam yaz, fotoğrafını çek ve yükle!

In [ ]:
try:
    from google.colab import files            # Colab'da dosya yükleme aracı
    yuklenen = files.upload()                 # "Dosya Seç" düğmesi çıkar
    for dosya_adi, icerik in yuklenen.items():
        foto = Image.open(io.BytesIO(icerik))
        tahmin_goster(resmi_hazirla(foto), f"{dosya_adi} (8x8 hali)")
except ImportError:
    print("⚠️ Dosya yükleme sadece Google Colab'da çalışır.")

> 🧐 **Model bazen yanılıyor mu?** Bu çok normal ve aslında çok önemli bir ders! Modelimiz, yıllar önce **başka bir ülkedeki** insanların el yazısıyla eğitildi. Senin el yazın, kalemin, çizim tarzın onlarınkinden farklı. Buna **veri kayması** (*data shift* — modelin eğitildiği veri ile gerçek hayattaki verinin farklı olması) denir.
>
> Gerçek yapay zekâ şirketleri bu yüzden **milyonlarca** farklı insandan veri toplar. Veri ne kadar çeşitliyse, model o kadar iyi genelleme yapar!

---
## 🏆 Meydan Okumalar (Challenge)

1. **🎯 Model Avcısı:** Çizim tuvalinde modeli **kandırmaya** çalış! Sen 3 çizdiğin halde 8 diyen bir çizim bulabilir misin? Hangi rakamlar daha kolay karışıyor? Karışıklık matrisiyle karşılaştır.
2. **🔧 Ayar Ustası:** `RandomForestClassifier` modelindeki `n_estimators` (ağaç sayısını) 5, 20, 100 yaparak dene. Ağaç sayısı azalınca doğruluk ne oluyor?
3. **🧪 Bilim İnsanı:** Eğitim verisini küçültürsen ne olur? `train_test_split` içinde `test_size=0.9` yap (yani sadece %10 eğitim verisi). Doğruluk ne kadar düşüyor? Bu bize **veri miktarı** hakkında ne anlatıyor?
4. **🕵️ Dedektif:** "Ortalama rakam" resimlerine tekrar bak. Sence model neden 1 ile 7'yi ya da 3 ile 8'i karıştırıyor olabilir?

In [ ]:
# Meydan okumalar için kodunu buraya yaz 👇

---
## 📝 Bugün Neler Öğrendik?
- 🖼️ Bilgisayar için bir resim, **piksellerden oluşan sayı tablosudur.**
- 📏 Resmi modele vermek için **düzleştiririz** (*flatten*): 8×8 → 64 özellik.
- ✂️ Modeli dürüstçe değerlendirmek için veriyi **eğitim** ve **test** olarak ayırırız.
- 🏘️ **KNN**, yeni örneğe en çok benzeyen komşulara bakarak karar verir.
- 🧩 **Karışıklık matrisi**, modelin hangi sınıfları karıştırdığını gösterir.
- 🎛️ **Hiperparametreler** (K, ağaç sayısı…) modelin başarısını etkiler.
- 🌍 Model sadece **gördüğü veriye benzer** şeylerde iyidir. Veri çeşitliliği çok önemlidir!

🎉 **Tebrikler, ilk görüntü tanıma yapay zekânı yaptın!**